In [25]:
import os
import tensorflow as tf # type: ignore
from tensorflow.keras.applications import ResNet50 # type: ignore
from tensorflow.keras.applications.resnet50 import preprocess_input # type: ignore
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout # type: ignore
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint # type: ignore
import imgaug.augmenters as iaa

In [26]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.config.optimizer.set_jit(True)

In [27]:
path = 'Dataset'
img_size = (200, 200)
batch_size = 64

In [28]:
seq = iaa.Sequential([
    iaa.Resize(img_size),
    iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0, 0.5))),
    iaa.Sometimes(0.5, iaa.AdditiveGaussianNoise(scale=(0, 0.05*255))),
    iaa.Sometimes(0.5, iaa.LinearContrast((0.75, 1.5))),
    iaa.Sometimes(0.5, iaa.Multiply((0.8, 1.2))),
    iaa.Sometimes(0.5, iaa.Affine(rotate=(-10, 10), shear=(-5, 5), scale=(0.8, 1.2)))
])

def augment_images(images):
    return seq(images=images) if images.ndim == 4 else seq(image=images)

In [29]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 
)

train_generator = train_datagen.flow_from_directory(
    path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 55845 images belonging to 29 classes.
Found 13954 images belonging to 29 classes.


In [30]:
train_dataset = tf.data.Dataset.from_generator(
    lambda: train_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, *img_size, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, train_generator.num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

validation_dataset = tf.data.Dataset.from_generator(
    lambda: validation_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, *img_size, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, validation_generator.num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

In [31]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(200, 200, 3))

In [32]:
base_model.trainable = False

In [33]:
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)  
x = Dropout(0.4)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [34]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
lr_plateau_cb = ReduceLROnPlateau(
    monitor='val_loss',   
    factor=0.5,           
    patience=3,           
    verbose=1,            
    min_lr=1e-7          
)

In [36]:
early_stopping_cb = EarlyStopping(
    monitor='val_loss',   
    patience=5,           
    restore_best_weights=True,  
    verbose=1            
)

In [37]:
model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=[lr_plateau_cb, early_stopping_cb]  
)

Epoch 1/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.6368 - loss: 1.2579

2025-02-04 14:00:16.333520: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1926', 12 bytes spill stores, 12 bytes spill loads

2025-02-04 14:01:28.463054: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1926', 8 bytes spill stores, 8 bytes spill loads



873/873 ━━━━━━━━━━━━━━━━━━━━ 376s 421ms/step - accuracy: 0.6370 - loss: 1.2572 - val_accuracy: 0.8166 - val_loss: 0.5532 - learning_rate: 0.0010
Epoch 2/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 361s 413ms/step - accuracy: 0.9299 - loss: 0.2282 - val_accuracy: 0.8218 - val_loss: 0.5468 - learning_rate: 0.0010
Epoch 3/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 351s 402ms/step - accuracy: 0.9472 - loss: 0.1602 - val_accuracy: 0.8096 - val_loss: 0.6568 - learning_rate: 0.0010
Epoch 4/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 349s 400ms/step - accuracy: 0.9599 - loss: 0.1194 - val_accuracy: 0.8173 - val_loss: 0.6806 - learning_rate: 0.0010
Epoch 5/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.9611 - loss: 0.1171
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
873/873 ━━━━━━━━━━━━━━━━━━━━ 343s 393ms/step - accuracy: 0.9611 - loss: 0.1171 - val_accuracy: 0.8360 - val_loss: 0.6280 - learning_rate: 0.0010
Epoch 6/10
873/873 ━━━━━━━━━━━━━━━━━━━━ 354s 405ms/step - accuracy: 0.9773 - l

In [38]:
for layer in base_model.layers[-10:]:
    layer.trainable = True

In [39]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [40]:
checkpoint_cb = ModelCheckpoint(
    'models/ResNet50.keras', 
    monitor='val_loss', 
    save_best_only=True,  
    verbose=1
)

In [41]:
model.fit(
    train_generator,
    epochs=15,
    validation_data=validation_generator,
    callbacks=[lr_plateau_cb, early_stopping_cb, checkpoint_cb] 
)

Epoch 1/15
873/873 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step - accuracy: 0.9522 - loss: 0.1479
Epoch 1: val_loss improved from inf to 0.53766, saving model to models/ResNet50.keras
873/873 ━━━━━━━━━━━━━━━━━━━━ 408s 452ms/step - accuracy: 0.9522 - loss: 0.1479 - val_accuracy: 0.8387 - val_loss: 0.5377 - learning_rate: 1.0000e-05
Epoch 2/15
873/873 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.9802 - loss: 0.0663
Epoch 2: val_loss improved from 0.53766 to 0.51329, saving model to models/ResNet50.keras
873/873 ━━━━━━━━━━━━━━━━━━━━ 383s 439ms/step - accuracy: 0.9802 - loss: 0.0663 - val_accuracy: 0.8552 - val_loss: 0.5133 - learning_rate: 1.0000e-05
Epoch 3/15
873/873 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - accuracy: 0.9872 - loss: 0.0453
Epoch 3: val_loss improved from 0.51329 to 0.48436, saving model to models/ResNet50.keras
873/873 ━━━━━━━━━━━━━━━━━━━━ 382s 438ms/step - accuracy: 0.9872 - loss: 0.0453 - val_accuracy: 0.8621 - val_loss: 0.4844 - learning_rate: 1.0000e-05
Epoch 4/15
873/873 ━━━━